# Optimization workflow

The handoff to a downstream model: what to give it, how to **weight** columns, how to
**disaggregate** results, and how to **reuse** a clustering.

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

result = tsam.aggregate(data, n_clusters=6, period_duration="1D")

## What your model needs

- **`cluster_representatives`** — the typical periods (your model's time series input).
- **`cluster_counts`** — how many real periods each stands for (its weight in the objective).
- **`cluster_assignments`** — which typical period each original period maps to.

In [ ]:
print("representatives:", result.cluster_representatives.shape)
print("counts:", dict(result.cluster_counts))
print("assignments (first 10):", result.cluster_assignments[:10])

## Weight columns by importance

Raise a column's weight to make the clustering reproduce it more accurately, at the others'
expense:

In [ ]:
weighted = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    weights={"GHI": 1.0, "T": 1.0, "Wind": 1.0, "Load": 20.0},
)
pd.DataFrame({"equal": result.accuracy.rmse, "Load x20": weighted.accuracy.rmse}).round(
    4
)

## Map results back

`disaggregate()` expands a per-typical-period decision onto the full timeline:

In [ ]:
model_output = result.cluster_representatives[["Load"]] * 0.5
full_year = result.disaggregate(model_output)
print(
    "typical-period output:", model_output.shape, "-> full timeline:", full_year.shape
)

## Reuse a clustering

Cluster once, then apply the **same** grouping to other data (a new scenario, say) without
re-running the clustering. It serialises to JSON:

In [ ]:
from tsam import ClusteringResult

clustering = result.clustering
print("reapplied:", clustering.apply(data).cluster_representatives.shape)

clustering.to_json("clustering.json")
reloaded = ClusteringResult.from_json("clustering.json").apply(data)
print("from JSON:", reloaded.cluster_representatives.shape)

---

* [Working with typical periods](working_with_typical_periods.ipynb) — the assignments in depth,
  including inter-period storage.
* [How long will this take?](runtime.ipynb) — budgeting the clustering step itself.